#### Train a model to distinguish colors

In [1]:
# Load necessary libraries
import csv
import os
import sys
import pandas as pd

from pathlib import Path
from sklearn.model_selection import train_test_split
import torch
from ultralytics import YOLO

In [3]:
# Load the indexed color dataset
dataset_csv_path = Path("color_dataset_index.csv").resolve()
color_df = pd.read_csv(dataset_csv_path)

# Keep only valid rows with existing image files
color_df = color_df.dropna(subset=["image_path", "color"]).copy()
color_df["image_path"] = color_df["image_path"].astype(str)
color_df["color"] = color_df["color"].astype(str).str.lower()
color_df = color_df[color_df["image_path"].map(lambda p: Path(p).exists())].reset_index(drop=True)

if color_df.empty:
    raise ValueError("No valid image rows found in color_dataset_index.csv")

# Stratified split so each color appears in train and val
train_df, val_df = train_test_split(
    color_df,
    test_size=0.2,
    random_state=42,
    stratify=color_df["color"]
 )

print(f"Total valid images: {len(color_df)}")
print(f"Train images: {len(train_df)}")
print(f"Val images: {len(val_df)}")
print("Classes:", sorted(color_df["color"].unique()))

Total valid images: 233
Train images: 186
Val images: 47
Classes: ['black', 'blue', 'brown', 'green', 'orange', 'red', 'violet', 'white', 'yellow']


In [4]:
import shutil

# Build YOLO classification folder structure:
# color_yolo_dataset/
#   train/<class_name>/*.jpg
#   val/<class_name>/*.jpg
yolo_color_root = Path("color_yolo_dataset").resolve()
if yolo_color_root.exists():
    shutil.rmtree(yolo_color_root)

for split_name in ["train", "val"]:
    split_df = train_df if split_name == "train" else val_df
    for _, row in split_df.iterrows():
        class_dir = yolo_color_root / split_name / row["color"]
        class_dir.mkdir(parents=True, exist_ok=True)

        src_path = Path(row["image_path"])
        # Add index prefix to avoid filename collisions across source folders
        dst_name = f"{len(list(class_dir.glob('*'))):06d}_{src_path.name}"
        shutil.copy2(src_path, class_dir / dst_name)

print(f"YOLO dataset created at: {yolo_color_root}")
print("Train class folders:", len(list((yolo_color_root / 'train').glob('*'))))
print("Val class folders:", len(list((yolo_color_root / 'val').glob('*'))))

YOLO dataset created at: D:\VMGP\monash\2026 unihack\TayUnihack2026\color_yolo_dataset
Train class folders: 9
Val class folders: 9


In [5]:
# Train YOLOv8 classifier on GPU if CUDA is available
device = 0 if torch.cuda.is_available() else "cpu"
if torch.cuda.is_available():
    print("CUDA enabled:", torch.cuda.get_device_name(0))
else:
    print("CUDA not available, training on CPU")

# You can switch to yolov8l-cls.pt if you want higher capacity
model = YOLO("yolov8n-cls.pt")

train_results = model.train(
    data=str(yolo_color_root),
    epochs=30,
    imgsz=224,
    batch=32,
    device=device,
    workers=4,
    project="runs_color_classifier",
    name="yolov8n_color",
    exist_ok=True
)

print("Training finished.")
train_results

CUDA not available, training on CPU
Ultralytics 8.4.21  Python-3.14.3 torch-2.10.0+cpu CPU (AMD Ryzen 9 7945HX with Radeon Graphics)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=D:\VMGP\monash\2026 unihack\TayUnihack2026\color_yolo_dataset, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n-cls.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov8n_color, nbs=64, nms=Fal

ultralytics.utils.metrics.ClassifyMetrics object with attributes:

confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x0000012FE745C1A0>
curves: []
curves_results: []
fitness: 0.7978723347187042
keys: ['metrics/accuracy_top1', 'metrics/accuracy_top5']
results_dict: {'metrics/accuracy_top1': 0.6382978558540344, 'metrics/accuracy_top5': 0.957446813583374, 'fitness': 0.7978723347187042}
save_dir: WindowsPath('D:/VMGP/monash/2026 unihack/TayUnihack2026/runs/classify/runs_color_classifier/yolov8n_color')
speed: {'preprocess': 0.0004957446629092335, 'inference': 1.6147617021256047, 'loss': 2.1276525543764866e-05, 'postprocess': 4.6808480042369756e-05}
task: 'classify'
top1: 0.6382978558540344
top5: 0.957446813583374